# Load Libraries

In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import pandas as pd
import joblib
from sklearn.preprocessing import OrdinalEncoder
from pathlib import Path
import os

# Load data 


In [7]:
file_path = "../temp_data/financial_featured.csv"
if not os.path.exists(file_path):
    file_path = "./model-development/temp_data/financial_featured.csv"
if not os.path.exists(file_path):
    file_path = "model-development/temp_data/financial_featured.csv"

df = pd.read_csv(file_path)

FileNotFoundError: [Errno 2] No such file or directory: 'model-development/temp_data/financial_featured.csv'

In [ ]:
# Calculate the feature
good_financial_condition = (
            (df["credit_score"] >= 700) & 
            (df["savings_to_income_ratio"] > 3.5) & 
            (df["debt_to_income_ratio"] < 3.0)
        ).astype(int)
y = good_financial_condition

In [ ]:
columns_to_drop = [
        "user_id", 
        "record_date", 
        "credit_score", 
        "savings_to_income_ratio", 
        "debt_to_income_ratio"
    ]
X = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [ ]:
# 3. Handle Categorical Features
categorical_cols = ["gender", "education_level", "employment_status", "job_title", "has_loan", "loan_type", "region"]
existing_cat_cols = [col for col in categorical_cols if col in X.columns]

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[existing_cat_cols] = encoder.fit_transform(X[existing_cat_cols])

MODEL_DIR = Path("./model-development/model")
if not MODEL_DIR.parent.exists():
    MODEL_DIR = Path("../model")   
joblib.dump(encoder, MODEL_DIR / "categorical_encoder.pkl")

In [ ]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit on XGBoost Model

In [ ]:
model = XGBClassifier()
model.fit(X_train, y_train)
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(model, MODEL_DIR / "financial_model.pkl")

# Evaluate the model

In [ ]:

y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification Report:', classification_report(y_test, y_pred))
print('Confusion Matrix:', confusion_matrix(y_test, y_pred))